In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph,START ,END

class State(TypedDict):
    message:str
    response:str


def chatbot(state:State):
    return{
        "response":f"You said: {state['message']}"
    }

graph=StateGraph(State)

graph.add_node("chatbot",chatbot)
graph.add_edge(START,"chatbot")
graph.add_edge("chatbot",END)


app=graph.compile()

result=app.invoke({
    "message":"HELLO!!!!",
    "response":""
})

print(result)

{'message': 'HELLO!!!!', 'response': 'You said: HELLO!!!!'}


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

from langgraph.prebuilt import ToolNode,tools_condition



#### #! STATE
class State(TypedDict):
    messages:Annotated[list,add_messages]


#### #! LLM

llm=ChatOpenAI(
    model="qwen/qwen3-4b-2507",
    base_url="http://127.0.0.1:1234/v1",
    api_key="dummy",
    temperature=0.7
)

#### #! TOOL

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    
    weather = {
        "mumbai": "32°C and humid",
        "delhi": "35°C and sunny",
        "london": "18°C and cloudy"
    }

    return weather.get(
        city.lower(),
        f"I don't have weather data for {city}"
    )

@tool
def calculator(a:float,b:float,operation:str)->float:
    """ 
    Perform a basic mathematical operation.

    operation must be:
    add, subtract, multiply, or divide
    """

    if operation=="add":
        return a+b

    elif operation=="substract":
        return a-b

    elif operation=="multiply":
        return a*b

    elif operation=="divide":
        return a/b

    else:
        raise ValueError("Unknown operation")


tools=[calculator,get_weather]


#### #! LLM+TOOLS

llm_with_tools=llm.bind_tools(tools)

#### #! LLM NODE

def chatbot(state: State):
    # System instructions defining what is permitted
    system_instruction = {
        "role": "system",
        "content": (
            "You are a restricted assistant. You must ONLY answer questions using your tools "
            "If the user requests anything else, you must "
            "politely decline to answer."
        )
    }
    
    # Prepend the system instructions to the conversation history
    messages_to_send = [system_instruction] + state['messages']
    
    response = llm_with_tools.invoke(messages_to_send)

    return {
        "messages": [response]
    }



#### #! CREATE GRAPH

graph=StateGraph(State)

graph.add_node("chatbot",chatbot)
graph.add_node("tools",ToolNode(tools))


#### #! ADD EDGE

graph.add_edge(START,"chatbot")

graph.add_conditional_edges("chatbot",tools_condition)
graph.add_edge("tools","chatbot")


#### #! COMPILE
app=graph.compile()


#### #! RUN

question=input("Enter Your Query: ")

result=app.invoke({
    "messages":[
        {
            'role':"user",
            "content":question
        }
    ]
})

for message in result['messages']:
    print(type(message).__name__)
    print(message.content)
    print()

HumanMessage
what is 22+3

AIMessage


ToolMessage
25.0

AIMessage
22 + 3 = 25.



In [30]:
#### #! VISUALIZE AS PNG
try:
    # This requires an internet connection to fetch the rendered image from the Mermaid API
    png_data = app.get_graph().draw_mermaid_png()
    with open("graph.png", "wb") as f:
        f.write(png_data)
    print("Graph visualization saved as 'graph.png'")
except Exception as e:
    print(f"Could not generate PNG: {e}")


Graph visualization saved as 'graph.png'
